In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/aellieme/warm_start_diff_models.git"
REF = None
REPO = Path("/content/warm_start_diff_models")
if not REPO.exists():
    args = ["git", "clone", REPO_URL, str(REPO)]
    if REF:
        args += ["--branch", REF]
    subprocess.run(args, check=True)

os.chdir(REPO)
assert (REPO / "src/requirements.txt").exists(), "Нужна актуальная версия проекта"
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.8.22"], check=True)
VENV = REPO / ".demo-venv"
if not VENV.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", str(VENV)], check=True)
PYTHON = str(VENV / "bin/python")
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
                "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "check", "--python", PYTHON], check=True)
from google.colab import drive
drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/exp_results_new")
os.environ["EXPERIMENT_OUTPUT_DIR"] = str(OUTPUT)
os.environ["EXPERIMENT_FILE_SUFFIX"] = "_new"
sys.path.insert(0, str(REPO / "demo"))
import workflow as experiment
import pandas as pd
from IPython.display import display

In [ ]:
MODEL = 'Random'
DATASETS = ["ml-1m", "amazon_Baby", "amazon_Toys_and_Games"]
MAXLENS = [50, 100]
DEVICE = 'cpu'
MAX_EPOCHS = 250
PATIENCE = 25
RUN_TUNING = True
PARAMETER_MODE = "auto"
ctx = experiment.configure(MODEL, OUTPUT, PYTHON, REPO)
CASES = list(experiment.cases(MODEL, DATASETS, MAXLENS))
CANDIDATES = {case: [dict(r["params"]) for r in experiment.presets(MODEL, *case)] for case in CASES}
MANUAL_PARAMS = {case: dict(experiment.presets(MODEL, *case)[0]["params"]) for case in CASES}
display(pd.DataFrame([{"dataset": d, "maxlen": length, **r} for d, length in CASES for r in experiment.presets(MODEL, d, length)]))

In [ ]:
experiment.preflight(ctx, DEVICE)
with experiment.exclusive(ctx):
    experiment.prepare(ctx, DATASETS)

## Тюнинг — ML-1M

In [ ]:
# Запустите эту ячейку только если нужен автоматический выбор параметров для ML-1M.
if MODEL in experiment.BASELINES:
    print("У этой модели нет тюнинга; перейдите к тренировке.")
elif RUN_TUNING:
    case = ('ml-1m', None)
    for params in CANDIDATES[case]:
        experiment.run(ctx, *case, params, "tuning", MAX_EPOCHS, PATIENCE, DEVICE, source="August candidate")

## Тюнинг — Amazon Baby

In [ ]:
# Запустите эту ячейку только если нужен автоматический выбор параметров для Amazon Baby.
if MODEL in experiment.BASELINES:
    print("У этой модели нет тюнинга; перейдите к тренировке.")
elif RUN_TUNING:
    case = ('amazon_Baby', None)
    for params in CANDIDATES[case]:
        experiment.run(ctx, *case, params, "tuning", MAX_EPOCHS, PATIENCE, DEVICE, source="August candidate")

## Тюнинг — Amazon Toys

In [ ]:
# Запустите эту ячейку только если нужен автоматический выбор параметров для Amazon Toys.
if MODEL in experiment.BASELINES:
    print("У этой модели нет тюнинга; перейдите к тренировке.")
elif RUN_TUNING:
    case = ('amazon_Toys_and_Games', None)
    for params in CANDIDATES[case]:
        experiment.run(ctx, *case, params, "tuning", MAX_EPOCHS, PATIENCE, DEVICE, source="August candidate")

In [ ]:
rows = experiment.tuning_rows(ctx)
display(pd.DataFrame([{"id": r["id"], "dataset": r["dataset"], "maxlen": r["maxlen"],
                       "status": r["status"], "params": r.get("selected_params", r["params"]),
                       **r.get("validation", {}), "error": r.get("error", "")} for r in rows]))

## Тренировка

In [ ]:
SELECTED = {
    case: experiment.choose(ctx, *case, mode=PARAMETER_MODE, manual=MANUAL_PARAMS.get(case))
    for case in CASES
}
display(pd.DataFrame([{"dataset": d, "maxlen": length, **choice} for (d, length), choice in SELECTED.items()]))

In [ ]:
# Пример auto: PARAMETER_MODE="auto", RUN_TUNING=True.
# Пример manual: PARAMETER_MODE="manual", RUN_TUNING=False, затем измените MANUAL_PARAMS для этого случая.
case = ('ml-1m', None)
choice = SELECTED[case]
print("ML-1M:", choice)
experiment.run(ctx, *case, choice["params"], "training", device=DEVICE, source=choice["source"])

In [ ]:
# Пример auto: PARAMETER_MODE="auto", RUN_TUNING=True.
# Пример manual: PARAMETER_MODE="manual", RUN_TUNING=False, затем измените MANUAL_PARAMS для этого случая.
case = ('amazon_Baby', None)
choice = SELECTED[case]
print("Amazon Baby:", choice)
experiment.run(ctx, *case, choice["params"], "training", device=DEVICE, source=choice["source"])

In [ ]:
# Пример auto: PARAMETER_MODE="auto", RUN_TUNING=True.
# Пример manual: PARAMETER_MODE="manual", RUN_TUNING=False, затем измените MANUAL_PARAMS для этого случая.
case = ('amazon_Toys_and_Games', None)
choice = SELECTED[case]
print("Amazon Toys:", choice)
experiment.run(ctx, *case, choice["params"], "training", device=DEVICE, source=choice["source"])

In [ ]:
display(pd.DataFrame(experiment.read_registry(ctx)))
for filename in ("experimental_results_new.xlsx", "experimental_results_new.md"):
    print(OUTPUT / "experiment_tables" / filename)